In [0]:
# Ruta exacta según tu imagen
ruta = "/Volumes/workspace/default/parcial/"

# Cargar cada archivo CSV y guardarlo como tabla en el esquema 'electiva'
spark.read.csv(ruta + "customers.csv", header=True, inferSchema=True).write.mode("overwrite").saveAsTable("electiva.customers")

spark.read.csv(ruta + "employees.csv", header=True, inferSchema=True).write.mode("overwrite").saveAsTable("electiva.employee")

spark.read.csv(ruta + "orders.csv", header=True, inferSchema=True).write.mode("overwrite").saveAsTable("electiva.orders")

spark.read.csv(ruta + "productcategories.csv", header=True, inferSchema=True).write.mode("overwrite").saveAsTable("electiva.product_categories")

spark.read.csv(ruta + "productsubcategories.csv", header=True, inferSchema=True).write.mode("overwrite").saveAsTable("electiva.product_sub_categories")

spark.read.csv(ruta + "products.csv", header=True, inferSchema=True).write.mode("overwrite").saveAsTable("electiva.products")

spark.read.csv(ruta + "vendors.csv", header=True, inferSchema=True).write.mode("overwrite").saveAsTable("electiva.vendor")

spark.read.csv(ruta + "vendorproduct.csv", header=True, inferSchema=True).write.mode("overwrite").saveAsTable("electiva.vendor_product")

Antes que todo primero debo crear las tablas a partir de los archivos CSV

In [0]:
%sql
SELECT 
    customerid,
    concat(firstname, ' ', lastname) AS fullname,
    concat(customerid, '-', upper(substr(firstname, 1, 2)), '-', lower(substr(lastname, 1, 2)), '-AW') AS user_name
FROM electiva.customers;

1-Se utilizó la función concat para unir las diferentes partes de las cadenas de texto solicitadas. Para cumplir con el requerimiento de la consulta: nombre de usuario, se aplicó substr para extraer exactamente los dos primeros caracteres de los nombres y apellidos, combinándolo con upper y lower para garantizar que el primer nombre quedara en mayúsculas y el apellido en minúsculas.

In [0]:
%sql
-- 1. Crear la tabla vacía
CREATE TABLE electiva.page_user (
    customerid INT,
    user_name STRING
);

-- 2. Insertar los datos calculados
INSERT INTO electiva.page_user
SELECT 
    customerid,
    concat(customerid, '-', upper(substr(firstname, 1, 2)), '-', lower(substr(lastname, 1, 2)), '-AW') AS user_name
FROM electiva.customers;

In [0]:
%sql
SELECT * FROM electiva.page_user;

2-La solución se estructuró en dos pasos: primero use la función “CREATE TABLE” para definir la estructura de la tabla en el esquema “Electiva” con sus tipos de dato (INT y STRING). Segundo, se usó INSERT INTO ... SELECT para poblar la tabla masivamente desde la tabla original de clientes, reutilizando la lógica de transformación de texto del punto 1 para generar y guardar el user_name directamente.

In [0]:
%sql
CREATE OR REPLACE VIEW electiva.`v_utl_ordenes_mitad_año` AS
WITH ordenes_agosto AS (
    -- Unimos las tablas necesarias para obtener toda la información
    SELECT 
        p.productname,      
        p.modelname,
        u.user_name,
        o.salesorderid,
        c.categoryname,     
        o.orderdate,
        o.linetotal
    FROM electiva.orders o
    JOIN electiva.page_user u ON o.customerid = u.customerid
    JOIN electiva.products p ON o.productid = p.productid
    JOIN electiva.product_sub_categories s ON p.subcategoryid = s.subcategoryid
    JOIN electiva.product_categories c ON s.categoryid = c.categoryid
    -- Usamos ILIKE como dice la pista para filtrar agosto de 2011
    WHERE CAST(o.orderdate AS STRING) ILIKE '2011-08-%'
)
SELECT 
    productname,
    modelname,
    user_name,
    salesorderid,
    categoryname,
    orderdate,
    linetotal,
    
    -- Calculamos el porcentaje de descuento usando CASE WHEN
    CASE 
        WHEN categoryname = 'Bikes' THEN 0.10
        WHEN categoryname = 'Clothing' THEN 0.15
        WHEN categoryname = 'Components' THEN 0.08
        WHEN categoryname = 'Accessories' THEN 0.20
        ELSE 0 
    END AS `descuento_mitad_año`,
    
    -- Calculamos el total con el descuento aplicado (linetotal - linetotal * descuento)
    linetotal - (linetotal * CASE 
        WHEN categoryname = 'Bikes' THEN 0.10
        WHEN categoryname = 'Clothing' THEN 0.15
        WHEN categoryname = 'Components' THEN 0.08
        WHEN categoryname = 'Accessories' THEN 0.20
        ELSE 0 
    END) AS total_descuentos_aplicados
FROM ordenes_agosto;

In [0]:
%sql
SELECT * FROM electiva.`v_utl_ordenes_mitad_año`;

3-Se construyó la visualización lógica (CREATE OR REPLACE VIEW) apoyada en una subconsulta (WITH) para ordenar el código. Las sentencias JOIN fueron necesarias para cruzar la información de las órdenes con la jerarquía de categorías de los productos. Se aplicó ILIKE para filtrar el texto de las fechas correspondientes a agosto de 2011. La lógica de descuentos se resolvió con CASE WHEN para asignar el porcentaje exacto según el nombre de la categoría, lo cual permitió hacer el cálculo aritmético del total final en la misma consulta.

In [0]:
%sql
CREATE OR REPLACE MATERIALIZED VIEW electiva.customer_class AS
WITH conteo_ordenes AS (
    -- Filtramos las órdenes del 2011 y contamos cuántas distintas hizo cada cliente
    SELECT 
        customerid,
        COUNT(DISTINCT salesorderid) AS ammount_of_orders
    FROM electiva.orders
    WHERE CAST(orderdate AS STRING) LIKE '2011-%'
    GROUP BY customerid
)
SELECT 
    co.ammount_of_orders,
    u.user_name,
    concat(c.firstname, ' ', c.lastname) AS customerfullname,
    -- Clasificamos según la cantidad de órdenes
    CASE 
        WHEN co.ammount_of_orders >= 3 THEN 'Oro'
        WHEN co.ammount_of_orders = 2 THEN 'Plata'
        ELSE 'Ocasional'
    END AS user_level,
    co.customerid
FROM conteo_ordenes co
JOIN electiva.page_user u ON co.customerid = u.customerid
JOIN electiva.customers c ON co.customerid = c.customerid;

In [0]:
%sql
SELECT * FROM electiva.customer_class;

4-Se usó CREATE MATERIALIZED VIEW para guardar el resultado de la consulta, optimizando su lectura futura al tratarse de una segmentación de clientes. se resolvió mediante COUNT(DISTINCT salesorderid) agrupado por cliente (GROUP BY), lo que garantiza contar órdenes únicas y no la cantidad de ítems comprados. Finalmente, la categorización ('Oro', 'Plata', 'Ocasional') se automatizó con un CASE WHEN basado en ese conteo previo.